# 2D Toy Inverse Reconstruction\nReconstruct the initial field with MAP optimization.

In [ ]:
import jax.numpy as jnp\nimport matplotlib.pyplot as plt\n\nfrom diffcosmo.fields import generate_gaussian_field, power_law_power_spectrum\nfrom diffcosmo.inference import reconstruct_map\nfrom diffcosmo.observe import apply_observation_model\nfrom diffcosmo.pm import evolve_pm\nfrom diffcosmo.utils import cross_correlation

In [ ]:
power_fn = lambda k: power_law_power_spectrum(k, amplitude=1.0, index=-2.0)\ngrid_shape = (32, 32)\nsim_config = {'n_steps': 6, 'dt': 0.1, 'checkpointing': 'none', 'cosmology': {'Omega_m': 0.3, 'init_disp_scale': 0.2}}\nprior_config = {'amplitude': 1.0, 'index': -2.0, 'lambda_prior': 1e-2, 'eps': 1e-6}\nobs_config = {'noise_std': 0.1, 'mask': None}\nopt_config = {'lr': 0.05, 'beta1': 0.9, 'beta2': 0.999, 'eps': 1e-8, 'max_iters': 120, 'early_stop_patience': 30, 'early_stop_min_delta': 1e-6}\n\ntheta_true = generate_gaussian_field(grid_shape, power_fn, seed=0)\ndelta_true = evolve_pm(theta_true, **{k: sim_config[k] for k in ['n_steps', 'dt', 'cosmology', 'checkpointing']})\ny_obs = apply_observation_model(delta_true, noise_std=obs_config['noise_std'], mask=obs_config['mask'], seed=1)\ntheta_init = generate_gaussian_field(grid_shape, power_fn, seed=1)\nresult = reconstruct_map(y_obs, theta_init, sim_config, prior_config, obs_config, opt_config)\ntheta_map = result['theta_map']

In [ ]:
print('Initial corr:', float(cross_correlation(theta_true, theta_init)))\nprint('Final corr:', float(cross_correlation(theta_true, theta_map)))\n\nfig, ax = plt.subplots(1, 3, figsize=(12, 4))\nax[0].imshow(theta_true, cmap='RdBu_r')\nax[0].set_title('True')\nax[1].imshow(theta_init, cmap='RdBu_r')\nax[1].set_title('Init')\nax[2].imshow(theta_map, cmap='RdBu_r')\nax[2].set_title('MAP')\nplt.tight_layout()